In [ ]:
import numpy as np
import pandas as pd
import os
import s3fs
import seaborn as sns
np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

FILE_KEY_S3 = "table_analyse_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df = pd.read_csv(file_in, sep=",")

df.head()

In [ ]:
# --- Mise en forme commune aux graphiques « version mise en forme » ---
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

COULEUR = "#2a6f97"          # couleur des barres
COULEUR_REPERE = "#c1121f"   # médiane, valeurs hors cadre, alertes
SOURCE_SIRENE = "Source : Insee, base Sirene, établissements actifs du code NAF 56.30Z (débits de boissons)"
SOURCE_POP = "Insee, populations municipales 2023"


def fmt_nombre(x, decimales=0):
    """Nombre au format français : espace pour les milliers, virgule décimale."""
    return f"{x:,.{decimales}f}".replace(",", "\u00a0").replace(".", ",")


def formateur(decimales=0):
    return FuncFormatter(lambda x, _: fmt_nombre(x, decimales))


def nouvelle_figure(figsize=(9, 5)):
    """Figure au style épuré, sans modifier le style global (les graphiques originaux restent inchangés)."""
    with sns.axes_style("whitegrid"):
        return plt.subplots(figsize=figsize)


def mettre_en_forme(ax, titre, sous_titre, xlabel, ylabel, source=SOURCE_SIRENE, decalage=0):
    """Titre, sous-titre, noms d'axes, source et habillage communs à tous les graphiques."""
    ax.annotate(titre, xy=(0, 1), xycoords="axes fraction", xytext=(0, 26 + decalage),
                textcoords="offset points", fontsize=14, fontweight="bold", va="bottom")
    ax.annotate(sous_titre, xy=(0, 1), xycoords="axes fraction", xytext=(0, 8 + decalage),
                textcoords="offset points", fontsize=10, color="0.35", va="bottom")
    ax.annotate(source, xy=(0, 0), xycoords="axes fraction", xytext=(0, -50),
                textcoords="offset points", fontsize=8, color="0.45", va="top")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax, left=True)
    return ax

## Stats univariées

### Nombre de bistrots par ville

In [ ]:
df.nb_etablissements.describe()

In [ ]:
sns.histplot(data=df, x="nb_etablissements")

In [ ]:
# Version mise en forme
s = df["nb_etablissements"].dropna()
mediane, moyenne = s.median(), s.mean()

fig, ax = nouvelle_figure((9, 5))
sns.histplot(x=s, bins=50, ax=ax, color=COULEUR, edgecolor="white", linewidth=0.5)
ax.axvline(mediane, color=COULEUR_REPERE, ls="--", lw=1.5, label=f"Médiane : {fmt_nombre(mediane)}")
ax.axvline(moyenne, color="0.2", ls=":", lw=1.5, label=f"Moyenne : {fmt_nombre(moyenne, 1)}")
ax.legend(frameon=False)
ax.xaxis.set_major_formatter(formateur())
ax.yaxis.set_major_formatter(formateur())

mettre_en_forme(
    ax,
    titre="Nombre de bistrots par commune",
    sous_titre=f"{fmt_nombre(len(s))} communes ; maximum : {fmt_nombre(s.max())} bistrots",
    xlabel="Nombre de bistrots dans la commune",
    ylabel="Nombre de communes",
)
plt.show()

In [ ]:
sns.histplot(data=df, x="nb_etablissements").set_yscale("log")

In [ ]:
# Version mise en forme
s = df["nb_etablissements"].dropna()

fig, ax = nouvelle_figure((9, 5))
sns.histplot(x=s, bins=50, ax=ax, color=COULEUR, edgecolor="white", linewidth=0.5)
ax.set_yscale("log")
ax.axvline(s.median(), color=COULEUR_REPERE, ls="--", lw=1.5, label=f"Médiane : {fmt_nombre(s.median())}")
ax.legend(frameon=False)
ax.xaxis.set_major_formatter(formateur())
ax.yaxis.set_major_formatter(formateur())

mettre_en_forme(
    ax,
    titre="Nombre de bistrots par commune (échelle logarithmique)",
    sous_titre="L'échelle log rend visibles les communes, peu nombreuses, qui comptent beaucoup de bistrots",
    xlabel="Nombre de bistrots dans la commune",
    ylabel="Nombre de communes (échelle log)",
)
plt.show()

In [ ]:
sns.histplot(data=df, x="nb_etablissements").set(xlim=(0, 50), ylim=(0, 1000))

In [ ]:
# Version mise en forme
s = df["nb_etablissements"].dropna()
XMAX, YMAX = 50, 1000          # même cadrage que le graphique original
s_zoom = s[s <= XMAX]

fig, ax = nouvelle_figure((10, 5))
# discrete=True : une barre par valeur entière (0, 1, 2... bistrots)
sns.histplot(x=s_zoom, discrete=True, ax=ax, color=COULEUR, edgecolor="white", linewidth=0.5)
ax.set(xlim=(-0.5, XMAX + 0.5), ylim=(0, YMAX))

# Les barres qui dépassent le cadre sont tronquées : on écrit leur vraie hauteur au-dessus
comptes = s_zoom.value_counts()
for valeur, n in comptes[comptes > YMAX].items():
    ax.annotate(fmt_nombre(n), xy=(valeur, YMAX), xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", rotation=90, fontsize=8, color=COULEUR_REPERE,
                annotation_clip=False)
ax.yaxis.set_major_formatter(formateur())

part = fmt_nombre(100 * len(s_zoom) / len(s))
mettre_en_forme(
    ax,
    titre=f"Nombre de bistrots par commune : zoom sur 0 à {XMAX} bistrots",
    sous_titre=f"{part} % des communes ; axe vertical coupé à {fmt_nombre(YMAX)} (valeurs réelles en rouge)",
    xlabel="Nombre de bistrots dans la commune",
    ylabel="Nombre de communes",
    decalage=35,  # remonte le titre au-dessus des étiquettes rouges
)
plt.show()

### Nombre de bistrots pour 1000 habitants par ville

In [ ]:
df["bistrots_pour_1000_hab"] = df["nb_etablissements"] / df["POPULATION"] * 1000

In [ ]:
sns.histplot(data=df, x="bistrots_pour_1000_hab")

In [ ]:
# Version mise en forme
s = df["bistrots_pour_1000_hab"].replace([np.inf, -np.inf], np.nan).dropna()
seuil = s.quantile(0.99)
s_aff = s[s <= seuil]
mediane = s.median()

fig, ax = nouvelle_figure((9, 5))
sns.histplot(x=s_aff, bins=50, ax=ax, color=COULEUR, edgecolor="white", linewidth=0.5)
ax.axvline(mediane, color=COULEUR_REPERE, ls="--", lw=1.5, label=f"Médiane : {fmt_nombre(mediane, 2)} pour 1 000 hab.")
ax.legend(frameon=False)
ax.xaxis.set_major_formatter(formateur(1))
ax.yaxis.set_major_formatter(formateur())

mettre_en_forme(
    ax,
    titre="Nombre de bistrots pour 1 000 habitants par commune",
    sous_titre=(f"{fmt_nombre(len(s))} communes ; les {fmt_nombre((s > seuil).sum())} communes au-delà "
                f"du 99e centile ({fmt_nombre(seuil, 1)} pour 1 000 hab.) ne sont pas affichées"),
    xlabel="Nombre de bistrots pour 1 000 habitants",
    ylabel="Nombre de communes",
    source=f"{SOURCE_SIRENE} ; {SOURCE_POP}",
)
plt.show()

### Âge moyen des bistrots

In [ ]:
df["cat_age_moyen"] = np.trunc(df["age_moyen_bistrots"]).astype("category")

In [ ]:
sns.histplot(data=df, x="cat_age_moyen")

In [ ]:
# Version mise en forme
age = df["age_moyen_bistrots"].dropna()
# même découpage que cat_age_moyen (âge tronqué à l'année), mais sur un axe numérique :
# les âges sans aucune commune apparaissent comme des trous au lieu d'être sautés
age_annees = np.trunc(age)
mediane = age.median()

fig, ax = nouvelle_figure((10, 5))
sns.histplot(x=age_annees, discrete=True, ax=ax, color=COULEUR, edgecolor="white", linewidth=0.3)
ax.axvline(mediane, color=COULEUR_REPERE, ls="--", lw=1.5, label=f"Médiane : {fmt_nombre(mediane, 1)} ans")
ax.legend(frameon=False)
ax.yaxis.set_major_formatter(formateur())

# Signale les âges aberrants (bistrots datés de 1900 dans Sirene, cf. note ci-dessous)
if age.max() >= 100:
    ax.annotate(f"{fmt_nombre(age.max())} ans : valeur aberrante probable\n(date de création 1900 dans Sirene)",
                xy=(np.trunc(age.max()), 0), xytext=(-10, 60), textcoords="offset points",
                ha="right", fontsize=9, color=COULEUR_REPERE,
                arrowprops=dict(arrowstyle="->", color=COULEUR_REPERE))

mettre_en_forme(
    ax,
    titre="Âge moyen des bistrots par commune",
    sous_titre=f"{fmt_nombre(len(age))} communes ; âge calculé au 1er mars 2026 à partir de la date de création",
    xlabel="Âge moyen des bistrots de la commune (en années révolues)",
    ylabel="Nombre de communes",
)
plt.show()

126 = tiré par les bistrots qui ont 1900 en date d'ouverture, à nettoyer @mathilde (AVANT le calcul de la moyenne)

### Niveau de vie 

### Taux de pauvreté

### Nombre de candidats au 2nd tour

### Indicateur 1

### Indicateur 2

### Indicateur 3

## Annexe : modifications apportées aux graphiques (versions « mises en forme »)

Chaque graphique original est conservé tel quel. Sa version mise en forme se trouve dans la cellule juste en dessous, qui commence par le commentaire `# Version mise en forme`. Ces cellules reposent sur une cellule de réglages communs, ajoutée juste après le chargement des données : thème seaborn, couleurs, format des nombres à la française et fonction `mettre_en_forme()`. Il faut donc l'exécuter avant elles.

**Pour tous les graphiques**
- Titre en gras et sous-titre qui donne l'effectif (nombre de communes) et le contexte de lecture.
- Axes renommés en français et en clair, unités précisées (bistrots, communes, bistrots pour 1 000 habitants, années).
- Nombres au format français : espace pour les milliers, virgule décimale.
- Source des données en bas à gauche (Sirene, NAF 56.30Z ; populations Insee pour le taux).
- Habillage allégé : grille horizontale seule, cadre supprimé, barres séparées par un fin liseré blanc, couleur unique.
- Taille de figure fixée pour une lecture confortable.

**1. Nombre de bistrots par commune (histogramme)**
- Lignes verticales de la médiane et de la moyenne, avec leurs valeurs en légende : l'écart entre les deux montre l'asymétrie de la distribution.
- 50 classes fixes : avec le découpage automatique sur une étendue aussi large, les barres étaient si fines qu'elles devenaient invisibles.
- Le maximum est indiqué dans le sous-titre.

**2. Nombre de bistrots par commune (échelle logarithmique)**
- Le titre et l'axe vertical précisent que l'échelle est logarithmique. Les graduations s'affichent en nombres entiers (1, 10, 100…) et non en puissances de 10.
- 50 classes fixes, comme pour le graphique 1.
- Ligne de la médiane ; le sous-titre explique l'intérêt de l'échelle log.

**3. Nombre de bistrots par commune (zoom 0–50)**
- Une barre par valeur entière (`discrete=True`). Dans l'original, les classes étaient calculées sur toute l'étendue des données puis recadrées, d'où des barres larges et peu lisibles dans le zoom.
- Le cadrage d'origine est conservé (0–50 bistrots, 0–1 000 communes). Les barres qui dépassent 1 000 sont tronquées : leur vraie hauteur est écrite en rouge au-dessus.
- Le sous-titre donne la part des communes couvertes par le zoom.

**4. Nombre de bistrots pour 1 000 habitants**
- Les communes à population nulle (taux infini) sont écartées.
- Les valeurs au-delà du 99e centile (souvent de très petites communes, où un seul bistrot suffit à faire grimper le taux) ne sont pas affichées, car elles écrasaient le reste de la distribution. Leur nombre et le seuil sont indiqués dans le sous-titre.
- 50 classes fixes : avec le découpage automatique, les barres étaient trop fines pour être lisibles.
- Ligne de la médiane, unité (bistrots pour 1 000 habitants) rappelée sur l'axe et dans la légende.

**5. Âge moyen des bistrots**
- Même découpage par année révolue que `cat_age_moyen`, mais sur un axe numérique. Avec la variable catégorielle, les âges sans aucune commune étaient sautés et l'axe n'était pas proportionnel.
- Ligne de la médiane en années ; le sous-titre rappelle la date de référence du calcul (1er mars 2026).
- Les âges moyens de 100 ans ou plus sont signalés par une flèche comme valeurs aberrantes probables (dates de création à 1900 dans Sirene, cf. la note sous le graphique original).